In [1]:
import pandas as pd
import pyarrow
import os
import geopandas as gpd
import folium
import requests
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import snowflake.connector

In [2]:
load_dotenv()
conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    warehouse="compute_wh",
    database="nyc_taxi",
    schema="surge_predictor"
)

In [3]:
cs = conn.cursor()

In [4]:
cs.execute("USE DATABASE nyc_taxi")
cs.execute("USE SCHEMA surge_predictor")
cs.execute("USE WAREHOUSE compute_wh")

In [5]:
cs.execute("SELECT CURRENT_ROLE()")
print(cs.fetchall())

[('ACCOUNTADMIN',)]


In [6]:
cs.execute('select count(*) from nyc_taxi.surge_predictor."fct_zone_features"')
print(cs.fetchall())

[(197160,)]


In [7]:
cs.execute("SHOW TABLES IN SCHEMA nyc_taxi.surge_predictor")
print(cs.fetchall())

[(datetime.datetime(2026, 8, 7, 15, 20, 43, 450000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'FHVHV_TRIPS', 'NYC_TAXI', 'SURGE_PREDICTOR', 'TABLE', '', '', 20835763, 857320448, 'ACCOUNTADMIN', '1', 'OFF', 'OFF', 'OFF', None, None, 'N', 'N', 'ROLE', 'N', 'N', 'N', 'N', 'N', 'N', 'OFF', 'OFF'), (datetime.datetime(2026, 8, 7, 16, 0, 38, 666000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'fct_zone_features', 'NYC_TAXI', 'SURGE_PREDICTOR', 'TRANSIENT', '', '', 197160, 2585600, 'ACCOUNTADMIN', '1', 'OFF', 'OFF', 'OFF', None, None, 'N', 'N', 'ROLE', 'N', 'N', 'N', 'N', 'N', 'N', 'OFF', 'OFF'), (datetime.datetime(2026, 8, 7, 15, 24, 26, 268000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'taxi_zone_lookup', 'NYC_TAXI', 'SURGE_PREDICTOR', 'TABLE', '', '', 265, 5120, 'ACCOUNTADMIN', '1', 'OFF', 'OFF', 'OFF', None, None, 'N', 'N', 'ROLE', 'N', 'N', 'N', 'N', 'N', 'N', 'OFF', 'OFF')]


In [8]:
cs.execute('''
select count(distinct zone_id) as zones, count(distinct pickup_hour_ts) as hours
from nyc_taxi.surge_predictor."fct_zone_features"
''')
print(cs.fetchall())

[(265, 744)]


In [10]:
cs.execute('DESCRIBE TABLE nyc_taxi.surge_predictor."taxi_zone_lookup";')print(cs.fetchall())

[('LOCATIONID', 'NUMBER(38,0)', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None, None), ('BOROUGH', 'VARCHAR(16777216)', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None, None), ('ZONE', 'VARCHAR(16777216)', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None, None), ('SERVICE_ZONE', 'VARCHAR(16777216)', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None, None)]


In [11]:
cs.execute("""
select count(*) as total_rows,
       count(distinct zone_id) as zones,
       count(distinct pickup_hour_ts) as hours
from nyc_taxi.surge_predictor."fct_zone_features"
""")
print(cs.fetchall())

[(197160, 265, 744)]
